# Analyze Long-Term Trends and Detect State Changes

In gravitational-wave observatory operations and environmental monitoring, low-frequency sensor records (e.g., minute trends of ground tilt, vacuum pressure, or thermal drift) span days to months. Distinguishing slow baseline drift from abrupt state steps and localized transient pulses is essential for data quality tracking.

**What you will achieve:**
1. Generate a 48-hour synthetic multi-channel trend dataset with known diurnal cycles, drift, pulses, steps, and gaps.
2. Estimate baseline trends across contiguous valid runs using low-pass filtering while respecting missing-data boundaries.
3. Isolate transient pulses and step changes using unit-aware signal analysis and derivative thresholding.
4. Export a structured event candidate table and validation metrics conforming to GWexpy data contracts.

**Data type**: Synthetic data generated in this notebook (48 h, 60 s cadence, 4 channels in Volts).

## Environment Setup and Output Configuration

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u
from scipy import signal

import gwexpy
from gwexpy.timeseries import TimeSeries, TimeSeriesDict

# Configure isolated artifact destination
output_dir = Path(os.environ.get("GWEXPY_DOCS_OUTPUT_DIR") or tempfile.mkdtemp(prefix="gwexpy-t1-"))
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "figures").mkdir(exist_ok=True)
(output_dir / "tables").mkdir(exist_ok=True)
print(f"Artifacts will be stored at: {output_dir}")

## Synthetic Trend Data Contract

We generate 48 hours of data sampled at $\Delta t = 60\,\text{s}$ ($N = 2880$ points per channel) starting at GPS $t_0 = 1400000000\,\text{s}$.
- `SYN:TREND_A`: 24-hour diurnal cycle + linear drift + 2 positive pulses at 8 h and 32 h.
- `SYN:TREND_B`: Diurnal cycle + drift + 2 positive pulses at 16 h and 40 h.
- `SYN:STATE`: Persistent baseline steps at 12 h (+0.6 V) and 36 h (-0.6 V).
- `SYN:GAP`: Same baseline as TREND_A but with a 20-minute gap (NaNs) between 22:00 and 22:20.

In [ ]:
def make_trend_fixture(seed: int = 2026091601):
    rng = np.random.default_rng(seed)
    n_samples = 2880
    dt_sec = 60.0
    t0_gps = 1400000000.0
    t_sec = np.arange(n_samples) * dt_sec
    t_hours = t_sec / 3600.0

    # Diurnal cycle (24 h period) and slow drift
    diurnal_a = 0.1 * np.sin(2 * np.pi * t_sec / 86400.0)
    drift_a = 0.0001 * t_hours
    baseline_a = diurnal_a + drift_a

    # Injected pulses in A: 8 h (sample 480), 32 h (sample 1920)
    pulse_a = np.zeros(n_samples)
    pulse_a[480] = 0.8
    pulse_a[1920] = 0.8
    data_a = baseline_a + pulse_a + rng.normal(0, 0.005, n_samples)

    # Injected pulses in B: 16 h (sample 960), 40 h (sample 2400)
    diurnal_b = 0.08 * np.cos(2 * np.pi * t_sec / 86400.0)
    drift_b = -0.00005 * t_hours
    baseline_b = diurnal_b + drift_b
    pulse_b = np.zeros(n_samples)
    pulse_b[960] = 0.8
    pulse_b[2400] = 0.8
    data_b = baseline_b + pulse_b + rng.normal(0, 0.005, n_samples)

    # Injected steps in STATE: 12 h (sample 720, +0.6 V), 36 h (sample 2160, -0.6 V)
    state = np.zeros(n_samples)
    state[720:] += 0.6
    state[2160:] -= 0.6
    data_state = state + rng.normal(0, 0.002, n_samples)

    # Gap channel: missing 20 samples from sample 1320 to 1340
    data_gap = data_a.copy()
    data_gap[1320:1340] = np.nan

    ts_dict = TimeSeriesDict({
        "SYN:TREND_A": TimeSeries(data_a, dt=60*u.s, t0=t0_gps*u.s, unit=u.V, name="SYN:TREND_A"),
        "SYN:TREND_B": TimeSeries(data_b, dt=60*u.s, t0=t0_gps*u.s, unit=u.V, name="SYN:TREND_B"),
        "SYN:STATE": TimeSeries(data_state, dt=60*u.s, t0=t0_gps*u.s, unit=u.V, name="SYN:STATE"),
        "SYN:GAP": TimeSeries(data_gap, dt=60*u.s, t0=t0_gps*u.s, unit=u.V, name="SYN:GAP"),
    })

    truth_events = [
        {"event_id": "P01", "channel": "SYN:TREND_A", "kind": "pulse", "sample_index": 480, "gps": t0_gps + 480*60, "delta_v": 0.8},
        {"event_id": "P02", "channel": "SYN:TREND_A", "kind": "pulse", "sample_index": 1920, "gps": t0_gps + 1920*60, "delta_v": 0.8},
        {"event_id": "P03", "channel": "SYN:TREND_B", "kind": "pulse", "sample_index": 960, "gps": t0_gps + 960*60, "delta_v": 0.8},
        {"event_id": "P04", "channel": "SYN:TREND_B", "kind": "pulse", "sample_index": 2400, "gps": t0_gps + 2400*60, "delta_v": 0.8},
        {"event_id": "S01", "channel": "SYN:STATE", "kind": "step", "sample_index": 720, "gps": t0_gps + 720*60, "delta_v": 0.6},
        {"event_id": "S02", "channel": "SYN:STATE", "kind": "step", "sample_index": 2160, "gps": t0_gps + 2160*60, "delta_v": -0.6},
    ]

    return ts_dict, truth_events

ts_dict, truth_events = make_trend_fixture()
print(f"Created TimeSeriesDict with {len(ts_dict)} channels:")
for ch, ts in ts_dict.items():
    print(f"  {ch}: {ts.shape[0]} samples, unit={ts.unit}, dt={ts.dt}")

## Baseline Estimation Across Contiguous Valid Runs

Missing data gaps must never be filled blindly with zeros before filtering, as doing so introduces massive Gibbs ringing and spurious transients. Instead, we identify contiguous runs of valid (non-NaN) samples and estimate the baseline independently on sufficiently long intervals.

In [ ]:
def contiguous_valid_runs(arr: np.ndarray, min_length: int = 120):
    isnan = np.isnan(arr)
    valid_runs = []
    in_run = False
    start = 0
    for idx, flag in enumerate(isnan):
        if not flag and not in_run:
            start = idx
            in_run = True
        elif flag and in_run:
            if idx - start >= min_length:
                valid_runs.append((start, idx))
            in_run = False
    if in_run and (len(arr) - start >= min_length):
        valid_runs.append((start, len(arr)))
    return valid_runs

def estimate_baseline(ts: TimeSeries, cutoff_hours: float = 1.0) -> TimeSeries:
    arr = ts.value.copy()
    baseline = np.full_like(arr, np.nan)
    fs = 1.0 / ts.dt.to(u.s).value
    f_cutoff = 1.0 / (cutoff_hours * 3600.0)
    sos = signal.butter(2, f_cutoff, btype="lowpass", fs=fs, output="sos")

    runs = contiguous_valid_runs(arr, min_length=int(cutoff_hours * 60 * 2))
    for start, end in runs:
        seg = arr[start:end]
        baseline[start:end] = signal.sosfiltfilt(sos, seg)

    return TimeSeries(baseline, dt=ts.dt, t0=ts.t0, unit=ts.unit, name=f"{ts.name}:BASELINE")

baselines = {}
for ch in ["SYN:TREND_A", "SYN:TREND_B", "SYN:GAP"]:
    baselines[ch] = estimate_baseline(ts_dict[ch])

# Export coverage table
coverage_records = []
for ch in ts_dict:
    arr = ts_dict[ch].value
    runs = contiguous_valid_runs(arr, min_length=1)
    for start, end in runs:
        coverage_records.append({
            "channel": ch,
            "start_sample": start,
            "end_sample": end,
            "duration_minutes": (end - start),
            "status": "valid"
        })
pd.DataFrame(coverage_records).to_csv(output_dir / "tables/coverage.csv", index=False)
print("Coverage table written to tables/coverage.csv")

## Detecting Transient Pulses and Persistent Steps

We detect two distinct classes of events:
1. **Transient Pulses**: Positive peaks in the residual signal `residual = raw - baseline` using `TimeSeries.find_peaks()`.
2. **Persistent Steps**: Run-length encoded transitions in the adjacent sample difference `diff = x[k] - x[k-1]` with $|\Delta V| > 0.3\,\text{V}$.

In [ ]:
detected_events = []
event_counter = 1

# 1. Pulses in Trend channels
for ch in ["SYN:TREND_A", "SYN:TREND_B"]:
    raw = ts_dict[ch]
    base = baselines[ch]
    residual_val = np.where(np.isnan(base.value), 0.0, raw.value - base.value)
    residual = TimeSeries(residual_val, dt=raw.dt, t0=raw.t0, unit=raw.unit, name=f"{ch}:RES")
    peaks, props = residual.find_peaks(height=0.3 * u.V, distance=30 * u.min)
    for p_val, t_val in zip(peaks.value, peaks.times.value):
        s_idx = int(round((t_val - raw.t0.value) / raw.dt.value))
        if residual_val[s_idx] > 0.3:
            detected_events.append({
                "event_id": f"EVT_{event_counter:03d}",
                "channel": ch,
                "kind": "pulse",
                "sample_index": s_idx,
                "event_gps_s": float(t_val),
                "peak_v": float(raw.value[s_idx]),
                "baseline_v": float(base.value[s_idx]),
                "delta_v": float(residual_val[s_idx]),
                "status": "candidate"
            })
            event_counter += 1

# 2. Steps in STATE channel
state_ts = ts_dict["SYN:STATE"]
diff_val = np.diff(state_ts.value)
step_indices = np.where(np.abs(diff_val) > 0.3)[0]
for idx in step_indices:
    sample_after = idx + 1
    t_val = state_ts.t0.value + sample_after * state_ts.dt.value
    detected_events.append({
        "event_id": f"EVT_{event_counter:03d}",
        "channel": "SYN:STATE",
        "kind": "step",
        "sample_index": sample_after,
        "event_gps_s": float(t_val),
        "peak_v": float(state_ts.value[sample_after]),
        "baseline_v": float(state_ts.value[idx]),
        "delta_v": float(diff_val[idx]),
        "status": "candidate"
    })
    event_counter += 1

events_df = pd.DataFrame(detected_events)
events_df.to_csv(output_dir / "tables/events.csv", index=False)
print(f"Extracted {len(events_df)} candidate events:")
print(events_df[["event_id", "channel", "kind", "sample_index", "delta_v"]])

## Visualization and Diagnostic Plots

In [ ]:
# Plot 1: Raw trend and estimated baseline
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
t_hours = (ts_dict["SYN:TREND_A"].times.value - ts_dict["SYN:TREND_A"].t0.value) / 3600.0

axs[0].plot(t_hours, ts_dict["SYN:TREND_A"].value, label="Raw SYN:TREND_A", color="tab:blue", alpha=0.6)
axs[0].plot(t_hours, baselines["SYN:TREND_A"].value, label="Baseline", color="navy", lw=1.5)
axs[0].set_ylabel("Signal [V]")
axs[0].legend(loc="upper right")
axs[0].grid(True, alpha=0.3)

axs[1].plot(t_hours, ts_dict["SYN:TREND_B"].value, label="Raw SYN:TREND_B", color="tab:green", alpha=0.6)
axs[1].plot(t_hours, baselines["SYN:TREND_B"].value, label="Baseline", color="darkgreen", lw=1.5)
axs[1].set_ylabel("Signal [V]")
axs[1].legend(loc="upper right")
axs[1].grid(True, alpha=0.3)

axs[2].plot(t_hours, ts_dict["SYN:GAP"].value, label="SYN:GAP (with missing data)", color="tab:orange", alpha=0.6)
axs[2].plot(t_hours, baselines["SYN:GAP"].value, label="Segmented Baseline", color="tab:red", lw=1.5)
axs[2].set_ylabel("Signal [V]")
axs[2].set_xlabel("Elapsed Time [hours]")
axs[2].legend(loc="upper right")
axs[2].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(output_dir / "figures/trend_baseline.png", dpi=150)
plt.close(fig)

# Plot 2: Transients and State Steps
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
res_a = ts_dict["SYN:TREND_A"].value - baselines["SYN:TREND_A"].value
ax1.plot(t_hours, res_a, label="Residual SYN:TREND_A", color="tab:purple")
for _, evt in events_df[events_df["channel"] == "SYN:TREND_A"].iterrows():
    ax1.axvline(evt["sample_index"] * 60.0 / 3600.0, color="crimson", ls="--", alpha=0.7)
ax1.set_ylabel("Residual [V]")
ax1.set_title("Detected Pulses")
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(t_hours, ts_dict["SYN:STATE"].value, label="SYN:STATE", color="tab:brown")
for _, evt in events_df[events_df["channel"] == "SYN:STATE"].iterrows():
    ax2.axvline(evt["sample_index"] * 60.0 / 3600.0, color="darkred", ls=":", lw=2)
ax2.set_ylabel("Voltage [V]")
ax2.set_xlabel("Elapsed Time [hours]")
ax2.set_title("Detected State Steps")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
fig.savefig(output_dir / "figures/transients_and_steps.png", dpi=150)
plt.close(fig)
print("Diagnostic figures saved to figures/")

## Verification Checks and Quality Metrics

We verify the results against the ground truth fixture and write the summary metrics to `validation-metrics.json`.

In [ ]:
# Check 1: Truth recovery
truth_recovered = True
for tr in truth_events:
    match = events_df[(events_df["channel"] == tr["channel"]) & (events_df["kind"] == tr["kind"])]
    sample_diffs = np.abs(match["sample_index"] - tr["sample_index"])
    if len(sample_diffs) == 0 or np.min(sample_diffs) > 1:
        truth_recovered = False
        break

# Check 2: Units
trend_units_ok = all(ts.unit == u.V for ts in ts_dict.values())

# Check 3: Gap policy (no events inside the gap)
gap_events = events_df[(events_df["channel"] == "SYN:GAP") & (events_df["sample_index"] >= 1320) & (events_df["sample_index"] <= 1340)]
gap_policy_ok = len(gap_events) == 0

# Check 4: Export roundtrip
loaded_df = pd.read_csv(output_dir / "tables/events.csv")
export_ok = len(loaded_df) == len(events_df)

metrics = {
    "status": "passed" if (truth_recovered and trend_units_ok and gap_policy_ok and export_ok) else "failed",
    "checks": {
        "trend_truth_recovery": {"passed": truth_recovered, "expected_events": 6, "observed_events": len(events_df)},
        "trend_units": {"passed": trend_units_ok, "unit": "V"},
        "trend_gap_policy": {"passed": gap_policy_ok, "gap_events_count": len(gap_events)},
        "trend_export_roundtrip": {"passed": export_ok, "rows": len(loaded_df)}
    }
}

with open(output_dir / "validation-metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

settings = {
    "tutorial_id": "T1",
    "duration_hours": 48,
    "sample_rate_hz": 1.0 / 60.0,
    "channels": list(ts_dict.keys()),
    "baseline_cutoff_hours": 1.0,
    "pulse_threshold_v": 0.3,
    "step_threshold_v": 0.3
}
with open(output_dir / "analysis-settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

print("Validation metrics:")
print(json.dumps(metrics, indent=2))
assert metrics["status"] == "passed", "Quality metrics check failed!"